# Africa Insect Occurrence Data Pipeline
## Cleaning, Validation & Spatial Analysis

**Workflow Overview:**
1. Data ingestion and exploration
2. Data quality assessment and cleaning
3. Spatial validation (coordinates, oceans, bounds)
4. Spatial bias analysis
5. Nearest-neighbor distance analysis
6. Sampling density surface generation
7. Multi-scale visualization
8. GeoPackage and raster export

**Outputs:**
- 6 publication-quality figures
- Clean occurrence GeoPackage (`clean_occurrences.gpkg`)
- Sampling density raster (`sampling_density.tif`)

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import Affine
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from scipy.stats import gaussian_kde, shapiro
from sklearn.neighbors import NearestNeighbors
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

# Set publication-quality defaults
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 10,
    'font.family': 'sans-serif',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'lines.linewidth': 1.2,
    'patch.linewidth': 0.5,
})

sns.set_palette("husl")
print("✓ Environment initialized")
print(f"  - GeoPandas {gpd.__version__}")
print(f"  - Rasterio {rasterio.__version__}")
print(f"  - Pandas {pd.__version__}")


✓ Environment initialized
  - GeoPandas 1.1.3
  - Rasterio 1.5.0
  - Pandas 3.0.3


In [5]:
# Try to load your data; if not found, generate synthetic dataset
DATA_PATH = r'/run/media/vincent/Extreme Pro/FaithAhiono/Species occurrence Africa/Species_occurrence/Species_occurrence/Butterfly_Moth.shp'  # Change to your file path

try:
    # Expected columns: species, latitude, longitude, date, uncertainty_m, collector
    df = gpd.read_file(DATA_PATH)
    print(f"✓ Loaded {len(df):,} records from {DATA_PATH}")
    synthetic = False
except FileNotFoundError:
    print(f"⚠ {DATA_PATH} not found. Generating synthetic dataset...")
    
    # Realistic synthetic insect occurrence data for Africa
    np.random.seed(42)
    n_records = 5000
    
    # African geographic extent
    africa_bounds = {
        'north': 37.5,
        'south': -34.8,
        'east': 51.5,
        'west': -17.5
    }
    
    species_list = ['Acilius sulcatus', 'Cybister tripunctatus', 'Dytiscus marginalis',
                    'Hydrophilus piceus', 'Helophorus brevipalpis', 'Laccobius minutus',
                    'Ochthebius marinus', 'Hydraena riparia', 'Elmis aenea', 'Limnius volckmari']
    
    df = pd.DataFrame({
        'species': np.random.choice(species_list, n_records),
        'latitude': np.random.uniform(africa_bounds['south'], africa_bounds['north'], n_records),
        'longitude': np.random.uniform(africa_bounds['west'], africa_bounds['east'], n_records),
        'date': pd.date_range('2000-01-01', periods=n_records, freq='12H'),
        'uncertainty_m': np.random.gamma(2, 2000, n_records),  # meters
        'collector': np.random.choice(['GBIF', 'iNaturalist', 'Museum', 'Field Survey'], n_records)
    })
    
    # Inject realistic patterns: more records near known research sites
    east_africa_mask = np.random.random(n_records) < 0.4
    df.loc[east_africa_mask, 'latitude'] += np.random.normal(0, 3, east_africa_mask.sum())
    df.loc[east_africa_mask, 'longitude'] += np.random.normal(35, 3, east_africa_mask.sum())
    
    # Add some coastal bias
    coastal_mask = np.random.random(n_records) < 0.2
    df.loc[coastal_mask, 'latitude'] = np.abs(df.loc[coastal_mask, 'latitude'])
    
    # Add duplicates (for testing duplicate removal)
    n_dupes = int(0.05 * n_records)
    dupes = df.sample(n_dupes).copy()
    df = pd.concat([df, dupes], ignore_index=True)
    
    synthetic = True
    print(f"✓ Generated synthetic dataset: {len(df):,} records")
    print(f"  Species: {df['species'].nunique()} taxa")
    print(f"  Collectors: {df['collector'].unique().tolist()}")

print(f"\nInitial dataset shape: {df.shape}")
print(f"Column info:\n{df.dtypes}")
print(f"\nFirst 5 records:\n{df.head()}")
print(f"\nMissing values:\n{df.isnull().sum()}")


✓ Loaded 108,843 records from /run/media/vincent/Extreme Pro/FaithAhiono/Species occurrence Africa/Species_occurrence/Species_occurrence/Butterfly_Moth.shp

Initial dataset shape: (108843, 5)
Column info:
latitude       float64
longitude      float64
scientific         str
common_nam         str
geometry      geometry
dtype: object

First 5 records:
    latitude  longitude              scientific            common_nam  \
0 -12.385247  37.237427      Hemiolaus caeculus      Azure Hairstreak   
1 -15.783571  35.006272  Junonia hierta cebrene  African Yellow Pansy   
2 -15.364770  35.304866           Junonia terea         Soldier Pansy   
3 -15.366912  35.299705       Tuxentius melaena             Black Pie   
4 -15.327222  35.320833    Euchrysops malathana     Common Smoky Blue   

                     geometry  
0  POINT (37.23743 -12.38525)  
1  POINT (35.00627 -15.78357)  
2  POINT (35.30487 -15.36477)  
3  POINT (35.29971 -15.36691)  
4  POINT (35.32083 -15.32722)  

Missing values:


In [7]:
print("="*70)
print("DATA CLEANING PIPELINE")
print("="*70)

# Step 1: Remove exact duplicates
initial_count = len(df)
df_clean = df.drop_duplicates(subset=['latitude', 'longitude'], 
                               keep='first').reset_index(drop=True)
dupes_removed = initial_count - len(df_clean)
print(f"\n1. DUPLICATE RECORDS")
print(f"   Records removed: {dupes_removed:,}")
print(f"   Remaining: {len(df_clean):,}")

# Step 2: Remove records with missing critical coordinates
initial_count = len(df_clean)
df_clean = df_clean.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
coords_removed = initial_count - len(df_clean)
print(f"\n2. MISSING COORDINATES")
print(f"   Records removed: {coords_removed:,}")
print(f"   Remaining: {len(df_clean):,}")

# Step 3: Remove invalid coordinates (outside [-90, 90] and [-180, 180])
initial_count = len(df_clean)
df_clean = df_clean[
    (df_clean['latitude'] >= -90) & (df_clean['latitude'] <= 90) &
    (df_clean['longitude'] >= -180) & (df_clean['longitude'] <= 180)
].reset_index(drop=True)
invalid_removed = initial_count - len(df_clean)
print(f"\n3. INVALID COORDINATE BOUNDS")
print(f"   Records removed: {invalid_removed:,}")
print(f"   Remaining: {len(df_clean):,}")
print(f"   Bounds: Lat [{df_clean['latitude'].min():.2f}, {df_clean['latitude'].max():.2f}]")
print(f"           Lon [{df_clean['longitude'].min():.2f}, {df_clean['longitude'].max():.2f}]")

# Step 4: Convert to GeoDataFrame for spatial operations
gdf = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['longitude'], df_clean['latitude']),
    crs='EPSG:4326'  # WGS84
)

print(f"\n4. CONVERTED TO GEODATAFRAME")
print(f"   CRS: {gdf.crs}")
print(f"   Records: {len(gdf):,}")


DATA CLEANING PIPELINE

1. DUPLICATE RECORDS
   Records removed: 0
   Remaining: 108,843

2. MISSING COORDINATES
   Records removed: 0
   Remaining: 108,843

3. INVALID COORDINATE BOUNDS
   Records removed: 0
   Remaining: 108,843
   Bounds: Lat [-34.81, 37.32]
           Lon [-17.46, 45.45]

4. CONVERTED TO GEODATAFRAME
   CRS: EPSG:4326
   Records: 108,843


In [8]:
print("\n5. OCEAN REMOVAL (Spatial Validation)")

# Africa boundary shapefile - simplified coastline
# Using a coarse Africa extent mask; for production, use: 
# https://www.naturalearthdata.com/ NE_50m_coastline or NE_10m_admin_0_countries

def get_africa_bounds():
    """Return simplified Africa boundary geometry."""
    from shapely.geometry import Polygon
    
    # Africa bounding box (very coarse - for production use proper coastline)
    africa_exterior = [
        (-17.5, 37.5),   # NW
        (51.5, 37.5),    # NE
        (51.5, -34.8),   # SE
        (-17.5, -34.8),  # SW
        (-17.5, 37.5)    # Close
    ]
    
    # Simplified: Create land polygon excluding major ocean areas
    # NOTE: For production, load from Natural Earth or GEBCO coastline
    africa = Polygon(africa_exterior)
    return africa

try:
    # Try to load land area shapefile
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    africa_land = world[world['continent'] == 'Africa']
    africa_geom = africa_land.geometry.unary_union
    print("   ✓ Using Natural Earth Africa boundary")
except:
    africa_geom = get_africa_bounds()
    print("   ⚠ Using simplified Africa bounding box (consider Natural Earth data)")

# Check which points are within Africa boundary
initial_count = len(gdf)
within_africa = gdf.geometry.within(africa_geom)
gdf = gdf[within_africa].reset_index(drop=True)
ocean_removed = initial_count - len(gdf)

print(f"   Records removed (likely ocean): {ocean_removed:,}")
print(f"   Remaining: {len(gdf):,}")
print(f"   Retention rate: {(len(gdf)/initial_count)*100:.1f}%")



5. OCEAN REMOVAL (Spatial Validation)
   ⚠ Using simplified Africa bounding box (consider Natural Earth data)
   Records removed (likely ocean): 3
   Remaining: 108,840
   Retention rate: 100.0%


In [9]:
print("\n6. COORDINATE UNCERTAINTY ASSESSMENT")

# Set default uncertainty if missing (assume 1km)
gdf['uncertainty_m'] = gdf['uncertainty_m'].fillna(1000)

print(f"   Uncertainty statistics (meters):")
print(f"     Mean:    {gdf['uncertainty_m'].mean():>10.0f} m")
print(f"     Median:  {gdf['uncertainty_m'].median():>10.0f} m")
print(f"     Max:     {gdf['uncertainty_m'].max():>10.0f} m")
print(f"     95th %ile: {gdf['uncertainty_m'].quantile(0.95):>10.0f} m")

# Define uncertainty thresholds
high_uncertainty_threshold = 50000  # 50 km
extreme_uncertainty_threshold = 100000  # 100 km

high_unc = (gdf['uncertainty_m'] >= high_uncertainty_threshold).sum()
extreme_unc = (gdf['uncertainty_m'] >= extreme_uncertainty_threshold).sum()

print(f"\n   Records with high uncertainty (≥50km): {high_unc:,} ({high_unc/len(gdf)*100:.1f}%)")
print(f"   Records with extreme uncertainty (≥100km): {extreme_unc:,} ({extreme_unc/len(gdf)*100:.1f}%)")

# Flag high-uncertainty records (optional: set flag_only=True to keep them)
flag_only = True  # Set to False to remove high-uncertainty records
if not flag_only:
    initial_count = len(gdf)
    gdf = gdf[gdf['uncertainty_m'] < high_uncertainty_threshold].reset_index(drop=True)
    removed = initial_count - len(gdf)
    print(f"\n   Records removed (high uncertainty): {removed:,}")
else:
    gdf['high_uncertainty'] = gdf['uncertainty_m'] >= high_uncertainty_threshold
    print(f"\n   ✓ High-uncertainty records flagged (not removed)")

print(f"   Final clean records: {len(gdf):,}")



6. COORDINATE UNCERTAINTY ASSESSMENT


KeyError: 'uncertainty_m'

In [ ]:
print("\n7. SPATIAL BIAS ASSESSMENT")

# Reproject to Africa-centered projection for accurate area calculations
# Using ESRI:102022 (Africa Albers Equal Area Conic)
gdf_proj = gdf.to_crs('ESRI:102022')

# Calculate 2-degree grid cells (to assess clustering)
gdf['lon_bin'] = pd.cut(gdf['longitude'], bins=np.arange(-20, 52, 2))
gdf['lat_bin'] = pd.cut(gdf['latitude'], bins=np.arange(-35, 38, 2))
grid_counts = gdf.groupby(['lon_bin', 'lat_bin']).size()

print(f"   Grid cells (2° x 2°): {len(grid_counts)}")
print(f"   Occupied cells: {(grid_counts > 0).sum()}")
print(f"   Empty cells: {(grid_counts == 0).sum()}")
print(f"   Max records per cell: {grid_counts.max()}")
print(f"   Gini coefficient (spatial evenness): {calculate_gini(grid_counts.values):.3f}")

# Per-country statistics
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    gdf_with_country = gpd.sjoin(gdf, world[['geometry', 'name']], how='left', predicate='within')
    country_counts = gdf_with_country.groupby('name').size().sort_values(ascending=False)
    print(f"\n   Countries with records: {len(country_counts)}")
    print(f"   Top 5 countries:")
    for country, count in country_counts.head(5).items():
        print(f"     {country:.<30} {count:>6,}")
except:
    print("   (Country lookup skipped)")

def calculate_gini(x):
    """Calculate Gini coefficient for spatial bias."""
    x = np.array(x)
    x = x[x > 0]  # Remove zeros
    sorted_x = np.sort(x)
    n = len(sorted_x)
    cumsum = np.cumsum(sorted_x)
    return (2 * np.sum((n + 1 - np.arange(1, n + 1)) * sorted_x)) / (n * cumsum[-1]) - (n + 1) / n

print(f"\n   ✓ Spatial bias assessment complete")


In [ ]:
print("\n8. NEAREST-NEIGHBOR DISTANCE ANALYSIS")

# For large datasets, sample for efficiency
sample_size = min(2000, len(gdf))
gdf_sample = gdf.sample(n=sample_size, random_state=42)

# Extract coordinates in projected CRS (meters)
coords_proj = np.array([gdf_sample.geometry.x, gdf_sample.geometry.y]).T

# Calculate nearest-neighbor distances using KDTree (efficient)
nbrs = NearestNeighbors(n_neighbors=2, algorithm='kd_tree').fit(coords_proj)
distances, indices = nbrs.kneighbors(coords_proj)

# First neighbor is self (distance 0), second is actual NN
nn_distances = distances[:, 1]  # in meters

print(f"   Sample size (for efficiency): {sample_size:,}")
print(f"\n   Nearest-neighbor distances (meters):")
print(f"     Mean:        {nn_distances.mean():>12,.0f} m")
print(f"     Median:      {nn_distances.median() if hasattr(nn_distances, 'median') else np.median(nn_distances):>12,.0f} m")
print(f"     Min:         {nn_distances.min():>12,.0f} m")
print(f"     Max:         {nn_distances.max():>12,.0f} m")
print(f"     95th %ile:   {np.percentile(nn_distances, 95):>12,.0f} m")

# Test for spatial randomness (Shapiro-Wilk on log-transformed distances)
log_distances = np.log10(nn_distances + 1)
stat, p_value = shapiro(log_distances)
print(f"\n   Spatial randomness test (Shapiro-Wilk on log-distances):")
print(f"     Test statistic: {stat:.4f}")
print(f"     p-value:        {p_value:.2e}")
if p_value < 0.05:
    print(f"     ✓ Non-random spatial pattern detected (clustered distribution)")
else:
    print(f"     Random spatial distribution")

# Store for visualization
gdf['nn_distance'] = np.nan
gdf.loc[gdf_sample.index, 'nn_distance'] = nn_distances


In [ ]:
print("\n9. SAMPLING DENSITY SURFACE GENERATION")

# Create continuous sampling density using kernel density estimation
# Using equal-area projection for accurate density

# Reproject to Africa Albers Equal Area
gdf_proj = gdf.to_crs('ESRI:102022')
coords = np.array([gdf_proj.geometry.x, gdf_proj.geometry.y]).T

# Define grid at 10 km resolution (adjust for study area)
bounds = gdf_proj.total_bounds  # [minx, miny, maxx, maxy]
resolution = 10000  # 10 km in meters

x_grid = np.arange(bounds[0], bounds[2], resolution)
y_grid = np.arange(bounds[1], bounds[3], resolution)
xx, yy = np.meshgrid(x_grid, y_grid)

# Flatten for KDE
grid_points = np.column_stack([xx.ravel(), yy.ravel()])

# Kernel Density Estimation
kde = gaussian_kde(coords.T, bw_method=0.1)
density = kde(grid_points.T)
density_grid = density.reshape(xx.shape)

# Apply Gaussian smoothing
density_smooth = gaussian_filter(density_grid, sigma=2)

print(f"   Grid resolution: {resolution/1000:.0f} km")
print(f"   Grid extent: {len(x_grid)} x {len(y_grid)} cells")
print(f"   Density range: {density_smooth.min():.2e} to {density_smooth.max():.2e}")

# Store for export
density_data = {
    'density': density_smooth,
    'xx': xx,
    'yy': yy,
    'bounds': bounds,
    'resolution': resolution,
    'crs': 'ESRI:102022'
}

print(f"   ✓ Sampling density surface generated")


In [ ]:
print("\n" + "="*70)
print("FIGURE 1: AFRICA OCCURRENCE MAP")
print("="*70)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111)

# Plot Africa background
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    africa = world[world['continent'] == 'Africa']
    africa.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.5, zorder=1)
except:
    pass

# Plot occurrences with uncertainty visualization
scatter = ax.scatter(
    gdf['longitude'], gdf['latitude'],
    c=gdf['uncertainty_m'],
    s=15,
    alpha=0.6,
    cmap='RdYlGn_r',
    norm=plt.matplotlib.colors.LogNorm(vmin=100, vmax=gdf['uncertainty_m'].max()),
    edgecolors='none',
    zorder=2,
    rasterized=True  # For publication-quality output
)

# Formatting
ax.set_xlim(-18, 52)
ax.set_ylim(-35, 38)
ax.set_xlabel('Longitude', fontsize=11, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=11, fontweight='bold')
ax.set_title('Insect Occurrences Across Africa (n={:,})'.format(len(gdf)), 
             fontsize=13, fontweight='bold', pad=15)

ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5, zorder=0)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.02, fraction=0.046)
cbar.set_label('Coordinate Uncertainty (m)', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figure_1_occurrence_map.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: figure_1_occurrence_map.png")
plt.show()


In [ ]:
print("\n" + "="*70)
print("FIGURE 2: SAMPLING DENSITY HEATMAP")
print("="*70)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111)

# Create custom colormap (white -> light blue -> dark blue -> red)
colors_list = ['#ffffff', '#e6f2ff', '#4da6ff', '#003366', '#8b0000']
n_bins = 100
cmap = LinearSegmentedColormap.from_list('density', colors_list, N=n_bins)

# Plot density surface
im = ax.contourf(density_data['xx'], density_data['yy'], density_data['density'],
                  levels=20, cmap=cmap, alpha=0.85, zorder=1)

# Overlay as heatmap
density_plot = ax.imshow(
    density_data['density'],
    extent=[density_data['bounds'][0], density_data['bounds'][2],
            density_data['bounds'][1], density_data['bounds'][3]],
    origin='lower',
    cmap=cmap,
    alpha=0.7,
    zorder=1
)

# Reproject back to WGS84 for display
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    africa = world[world['continent'] == 'Africa']
    africa_4326 = africa.to_crs('EPSG:4326')
    africa_4326.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1, zorder=3)
except:
    pass

ax.set_xlim(-18, 52)
ax.set_ylim(-35, 38)
ax.set_xlabel('Longitude', fontsize=11, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=11, fontweight='bold')
ax.set_title('Sampling Density Surface (10-km resolution KDE)', 
             fontsize=13, fontweight='bold', pad=15)

# Colorbar
cbar = plt.colorbar(density_plot, ax=ax, pad=0.02, fraction=0.046)
cbar.set_label('Sampling Density (records/cell)', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figure_2_sampling_density.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure_2_sampling_density.png")
plt.show()


In [ ]:
print("\n" + "="*70)
print("FIGURE 3: COUNTRY-LEVEL OCCURRENCE COUNTS")
print("="*70)

# Spatial join with country boundaries
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    gdf_country = gpd.sjoin(gdf, world[['geometry', 'name']], how='left', predicate='within')
    country_counts = gdf_country.groupby('name').size().sort_values(ascending=True).tail(15)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(range(len(country_counts)), country_counts.values, 
                   color=plt.cm.viridis(np.linspace(0, 1, len(country_counts))))
    
    ax.set_yticks(range(len(country_counts)))
    ax.set_yticklabels(country_counts.index, fontsize=10)
    ax.set_xlabel('Number of Records', fontsize=11, fontweight='bold')
    ax.set_title('Top 15 Countries by Insect Occurrence Records', 
                 fontsize=12, fontweight='bold', pad=15)
    
    # Add value labels
    for i, v in enumerate(country_counts.values):
        ax.text(v + 5, i, str(v), va='center', fontsize=9, fontweight='bold')
    
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.savefig('figure_3_country_counts.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: figure_3_country_counts.png")
    plt.show()
except Exception as e:
    print(f"⚠ Country analysis skipped: {e}")


In [ ]:
print("\n" + "="*70)
print("FIGURE 4: BIOME-LEVEL OCCURRENCE COUNTS")
print("="*70)

# Classify by latitude-based biome proxy (simplified)
def classify_biome(lat):
    """Simple biome classification based on latitude."""
    if lat > 23.5:
        return 'Northern Temperate'
    elif lat > 10:
        return 'Sahel/Semi-Arid'
    elif lat > 0:
        return 'Guinean Forest-Savanna'
    elif lat > -10:
        return 'Congolian Rainforest'
    elif lat > -18:
        return 'Miombo Woodland'
    else:
        return 'Southern Temperate'

gdf['biome'] = gdf['latitude'].apply(classify_biome)
biome_counts = gdf['biome'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_biome = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']
bars = ax.barh(range(len(biome_counts)), biome_counts.values, color=colors_biome[:len(biome_counts)])

ax.set_yticks(range(len(biome_counts)))
ax.set_yticklabels(biome_counts.index, fontsize=11)
ax.set_xlabel('Number of Records', fontsize=11, fontweight='bold')
ax.set_title('Occurrences by African Biome (Latitude-based Classification)', 
             fontsize=12, fontweight='bold', pad=15)

# Add value labels and percentages
total = biome_counts.sum()
for i, v in enumerate(biome_counts.values):
    pct = (v / total) * 100
    ax.text(v + 5, i, f'{v:,} ({pct:.1f}%)', va='center', fontsize=9, fontweight='bold')

ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('figure_4_biome_counts.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure_4_biome_counts.png")
plt.show()


In [ ]:
print("\n" + "="*70)
print("FIGURE 5: LATITUDINAL DISTRIBUTION")
print("="*70)

fig = plt.figure(figsize=(14, 6))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

# Plot 1: Histogram with KDE
ax1 = fig.add_subplot(gs[0, :])
n, bins, patches = ax1.hist(gdf['latitude'], bins=50, color='#3498db', 
                             alpha=0.7, edgecolor='black', linewidth=0.5)

# Add KDE overlay
from scipy.stats import gaussian_kde as kde_func
kde_lat = kde_func(gdf['latitude'])
x_range = np.linspace(gdf['latitude'].min(), gdf['latitude'].max(), 200)
kde_values = kde_lat(x_range)
ax1_twin = ax1.twinx()
ax1_twin.plot(x_range, kde_values, 'r-', linewidth=2, label='KDE')
ax1_twin.set_ylabel('Density', fontsize=10, fontweight='bold')

ax1.set_xlabel('Latitude (degrees)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax1.set_title('Latitudinal Distribution of Insect Occurrences', 
              fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Plot 2: By latitude band
ax2 = fig.add_subplot(gs[1, 0])
lat_bands = pd.cut(gdf['latitude'], bins=np.arange(-40, 40, 5))
lat_band_counts = lat_bands.value_counts().sort_index()
ax2.bar(range(len(lat_band_counts)), lat_band_counts.values, color='#e74c3c', alpha=0.7)
ax2.set_xticks(range(len(lat_band_counts)))
ax2.set_xticklabels([f'{int(x.left)}°' for x in lat_band_counts.index], fontsize=8, rotation=45)
ax2.set_ylabel('Record Count', fontsize=10, fontweight='bold')
ax2.set_title('Records by 5° Latitude Band', fontsize=11, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Statistics by latitude zone
ax3 = fig.add_subplot(gs[1, 1])
ax3.axis('off')

zones = [
    ('Northern Temp.', 23.5, 90),
    ('Sahel/Semi-Arid', 10, 23.5),
    ('Guinea Forest', 0, 10),
    ('Congo Rainforest', -10, 0),
    ('Miombo Woodland', -18, -10),
    ('Southern Temp.', -90, -18),
]

stats_text = "Latitudinal Zone Statistics\n" + "="*35 + "\n"
for zone_name, lat_min, lat_max in zones:
    count = len(gdf[(gdf['latitude'] >= lat_min) & (gdf['latitude'] < lat_max)])
    pct = (count / len(gdf)) * 100 if len(gdf) > 0 else 0
    stats_text += f"{zone_name:.<20} {count:>6,} ({pct:>5.1f}%)\n"

ax3.text(0.05, 0.95, stats_text, transform=ax3.transAxes, fontsize=9,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.savefig('figure_5_latitudinal_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure_5_latitudinal_distribution.png")
plt.show()


In [ ]:
print("\n" + "="*70)
print("FIGURE 6: NEAREST-NEIGHBOR DISTANCE DISTRIBUTION")
print("="*70)

nn_data = gdf['nn_distance'].dropna()

fig = plt.figure(figsize=(14, 6))
gs = GridSpec(1, 2, figure=fig, hspace=0.3, wspace=0.3)

# Plot 1: Distribution on linear scale
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(nn_data / 1000, bins=50, color='#9b59b6', alpha=0.7, edgecolor='black', linewidth=0.5)
ax1.axvline(nn_data.mean() / 1000, color='red', linestyle='--', linewidth=2, label='Mean')
ax1.axvline(np.median(nn_data) / 1000, color='green', linestyle='--', linewidth=2, label='Median')
ax1.set_xlabel('Nearest-Neighbor Distance (km)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax1.set_title('NN Distance Distribution (Linear Scale)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Distribution on log scale
ax2 = fig.add_subplot(gs[0, 1])
log_distances = np.log10(nn_data)
ax2.hist(log_distances, bins=50, color='#1abc9c', alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.axvline(np.log10(nn_data.mean()), color='red', linestyle='--', linewidth=2, label='Mean (log)')
ax2.set_xlabel('Log₁₀(Distance) [log₁₀(meters)]', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('NN Distance Distribution (Log Scale)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.savefig('figure_6_nn_distance_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure_6_nn_distance_distribution.png")
plt.show()

print(f"\n" + "="*70)
print("FIGURES GENERATION COMPLETE")
print("="*70)


In [ ]:
print("\n" + "="*70)
print("EXPORT: CLEAN OCCURRENCES GEOPACKAGE")
print("="*70)

# Prepare export GeoDataFrame
export_gdf = gdf.copy()

# Ensure geometry is in WGS84
export_gdf = export_gdf.to_crs('EPSG:4326')

# Select and organize columns
export_columns = ['species', 'latitude', 'longitude', 'date', 'uncertainty_m', 
                  'collector', 'biome', 'geometry']
export_gdf = export_gdf[[col for col in export_columns if col in export_gdf.columns]]

# Export to GeoPackage
output_gpkg = 'clean_occurrences.gpkg'
export_gdf.to_file(output_gpkg, layer='occurrences', driver='GPKG')

print(f"✓ Exported to {output_gpkg}")
print(f"  Records: {len(export_gdf):,}")
print(f"  Columns: {list(export_gdf.columns)}")
print(f"  Spatial extent:")
print(f"    Latitude:  [{export_gdf.geometry.y.min():.2f}, {export_gdf.geometry.y.max():.2f}]")
print(f"    Longitude: [{export_gdf.geometry.x.min():.2f}, {export_gdf.geometry.x.max():.2f}]")

# Also export summary statistics layer
summary_layer = pd.DataFrame({
    'metric': ['Total records', 'Unique species', 'Spatial extent (km²)', 
               'Avg uncertainty (m)', 'NN distance mean (m)'],
    'value': [
        len(export_gdf),
        export_gdf['species'].nunique() if 'species' in export_gdf.columns else 0,
        'See figures for spatial extent',
        f"{export_gdf['uncertainty_m'].mean():.0f}",
        f"{nn_data.mean():.0f}",
    ]
})

print(f"\nSummary statistics:")
print(summary_layer.to_string(index=False))


In [ ]:
print("\n" + "="*70)
print("EXPORT: SAMPLING DENSITY RASTER")
print("="*70)

# Create GeoTIFF of sampling density
output_tif = 'sampling_density.tif'

# Get bounds and resolution from density data
bounds = density_data['bounds']
resolution = density_data['resolution']

# Create affine transform
transform = Affine.translation(bounds[0], bounds[3]) * Affine.scale(resolution, -resolution)

# Write to GeoTIFF
with rasterio.open(
    output_tif,
    'w',
    driver='GTiff',
    height=density_data['density'].shape[0],
    width=density_data['density'].shape[1],
    count=1,
    dtype=density_data['density'].dtype,
    transform=transform,
    crs='ESRI:102022',
    compress='lzw',
    nodata=0
) as dst:
    dst.write(density_data['density'], 1)

print(f"✓ Exported to {output_tif}")
print(f"  Projection: ESRI:102022 (Africa Albers Equal Area Conic)")
print(f"  Resolution: {resolution/1000:.0f} km")
print(f"  Dimensions: {density_data['density'].shape[0]} x {density_data['density'].shape[1]} pixels")
print(f"  Data range: {density_data['density'].min():.2e} to {density_data['density'].max():.2e}")
print(f"  Compression: LZW")


In [ ]:
print("\n" + "="*70)
print("DATA PIPELINE SUMMARY REPORT")
print("="*70)

summary_stats = {
    'Initial Records': initial_count,
    'Duplicates Removed': dupes_removed,
    'Missing Coordinates Removed': coords_removed,
    'Invalid Coordinates Removed': invalid_removed,
    'Ocean Records Removed': ocean_removed,
    'Final Clean Records': len(gdf),
    'Retention Rate (%)': f"{(len(gdf)/initial_count)*100:.1f}%",
    '': '',
    'Unique Species': gdf['species'].nunique() if 'species' in gdf.columns else 'N/A',
    'Latitude Range': f"{gdf['latitude'].min():.2f}° to {gdf['latitude'].max():.2f}°",
    'Longitude Range': f"{gdf['longitude'].min():.2f}° to {gdf['longitude'].max():.2f}°",
    'Mean Uncertainty (m)': f"{gdf['uncertainty_m'].mean():.0f}",
    'Median Uncertainty (m)': f"{gdf['uncertainty_m'].median():.0f}",
    '': '',
    'NN Distance Mean (m)': f"{nn_data.mean():.0f}",
    'NN Distance Median (m)': f"{np.median(nn_data):.0f}",
    'Spatial Clustering': 'Detected' if p_value < 0.05 else 'Random',
}

print("\nKey Statistics:")
for key, value in summary_stats.items():
    if key == '':
        print()
    else:
        print(f"  {key:.<45} {value}")

print(f"\nOutput Files Generated:")
print(f"  ✓ figure_1_occurrence_map.png")
print(f"  ✓ figure_2_sampling_density.png")
print(f"  ✓ figure_3_country_counts.png")
print(f"  ✓ figure_4_biome_counts.png")
print(f"  ✓ figure_5_latitudinal_distribution.png")
print(f"  ✓ figure_6_nn_distance_distribution.png")
print(f"  ✓ clean_occurrences.gpkg")
print(f"  ✓ sampling_density.tif")

print(f"\n{'='*70}")
print(f"Pipeline execution complete. All outputs ready for publication.")
print(f"{'='*70}")
